# Observer Pattern
## Design Patterns for the AI Era · Part 4

> **Bad Code → Better Design → Working Python → AI Application**

Decouple events from listeners. One agent action, many independent consumers.

## 1. The Problem

An agent calls tools. Every consumer (logger, metrics, audit, human review) is wired directly into the agent.

In [ ]:
# ❌ BAD CODE — tightly coupled agent

class Logger:
    def log(self, msg): print(f'[LOG] {msg}')

class MetricsCollector:
    def record(self, name): print(f'[METRICS] +1 {name}')

class AuditLog:
    def write(self, name, args): print(f'[AUDIT] {name}({args})')

class BadAgent:
    def __init__(self):
        # Directly instantiates every consumer — tight coupling
        self.logger  = Logger()
        self.metrics = MetricsCollector()
        self.audit   = AuditLog()

    def run_tool(self, tool_name: str, args: dict):
        result = f'result_of_{tool_name}'
        # Must call every consumer manually — adding one means editing Agent
        self.logger.log(f'{tool_name} called')
        self.metrics.record(tool_name)
        self.audit.write(tool_name, args)
        return result

agent = BadAgent()
agent.run_tool('search', {'query': 'factory pattern'})

## 2. What's Wrong?

- Agent knows about Logger, MetricsCollector, AuditLog — tight coupling
- Cannot test Agent without all three consumers
- Adding human review queue means editing `BadAgent`
- Consumers cannot be swapped (e.g., switch to cloud metrics)

In [ ]:
# ✅ GOOD CODE — Observer / EventBus
from dataclasses import dataclass, field
from typing import Callable, Any

@dataclass
class AgentEvent:
    kind: str
    data: dict[str, Any]

Observer = Callable[[AgentEvent], None]

class EventBus:
    def __init__(self):
        self._listeners: list[Observer] = []

    def subscribe(self, observer: Observer) -> None:
        self._listeners.append(observer)

    def unsubscribe(self, observer: Observer) -> None:
        self._listeners = [o for o in self._listeners if o is not observer]

    def publish(self, event: AgentEvent) -> None:
        for obs in self._listeners:
            obs(event)


class Agent:
    def __init__(self, bus: EventBus):
        self._bus = bus  # Only dependency: the event bus

    def run_tool(self, tool_name: str, args: dict):
        result = f'result_of_{tool_name}'
        self._bus.publish(AgentEvent(
            kind='tool_call',
            data={'tool': tool_name, 'args': args, 'result': result}
        ))
        return result


# Observers — completely independent of each other and of Agent
def logger_observer(event: AgentEvent) -> None:
    print(f'[LOG] {event.kind}: {event.data["tool"]}')

def metrics_observer(event: AgentEvent) -> None:
    if event.kind == 'tool_call':
        print(f'[METRICS] +1 {event.data["tool"]}')

def audit_observer(event: AgentEvent) -> None:
    print(f'[AUDIT] {event.data["tool"]}({event.data["args"]})')


# Wire up
bus = EventBus()
bus.subscribe(logger_observer)
bus.subscribe(metrics_observer)
bus.subscribe(audit_observer)

agent = Agent(bus)
agent.run_tool('search', {'query': 'factory pattern'})
print()
agent.run_tool('send_email', {'to': 'user@example.com'})

## 4. Tests

In [ ]:
# ✅ Tests

def test_observer_receives_events():
    received = []
    bus = EventBus()
    bus.subscribe(lambda e: received.append(e))

    agent = Agent(bus)
    agent.run_tool('search', {'query': 'test'})

    assert len(received) == 1
    assert received[0].kind == 'tool_call'
    assert received[0].data['tool'] == 'search'
    print('PASS test_observer_receives_events')

def test_multiple_observers():
    log_a, log_b = [], []
    bus = EventBus()
    bus.subscribe(lambda e: log_a.append(e.kind))
    bus.subscribe(lambda e: log_b.append(e.kind))

    Agent(bus).run_tool('tool_x', {})

    assert log_a == ['tool_call']
    assert log_b == ['tool_call']
    print('PASS test_multiple_observers')

def test_agent_works_with_no_observers():
    bus = EventBus()
    result = Agent(bus).run_tool('noop', {})
    assert result == 'result_of_noop'
    print('PASS test_agent_works_with_no_observers')

def test_unsubscribe():
    received = []
    bus = EventBus()
    observer = lambda e: received.append(e)
    bus.subscribe(observer)
    bus.unsubscribe(observer)
    Agent(bus).run_tool('x', {})
    assert received == []
    print('PASS test_unsubscribe')

test_observer_receives_events()
test_multiple_observers()
test_agent_works_with_no_observers()
test_unsubscribe()
print('\nAll tests passed ✅')

## 5. AI Example — Human-in-the-Loop Review Queue

In [ ]:
# 🤖 AI EXAMPLE — Human review for sensitive tool calls

SENSITIVE_TOOLS = {'send_email', 'delete_record', 'transfer_funds', 'modify_permissions'}

class HumanReviewQueue:
    def __init__(self):
        self._queue: list[dict] = []

    def push(self, item: dict) -> None:
        self._queue.append(item)
        print(f'[REVIEW QUEUE] Flagged for human approval: {item["tool"]}')

    def pending(self) -> list[dict]:
        return list(self._queue)


review_queue = HumanReviewQueue()

def human_review_observer(event: AgentEvent) -> None:
    if event.kind == 'tool_call' and event.data['tool'] in SENSITIVE_TOOLS:
        review_queue.push({
            'tool': event.data['tool'],
            'args': event.data['args'],
            'requires_approval': True
        })


bus2 = EventBus()
bus2.subscribe(logger_observer)
bus2.subscribe(human_review_observer)

agent2 = Agent(bus2)
agent2.run_tool('search', {'query': 'invoices'})          # safe
print()
agent2.run_tool('send_email', {'to': 'cfo@company.com'})  # flagged
print()
agent2.run_tool('delete_record', {'id': 'REC-001'})        # flagged

print('\nPending reviews:', len(review_queue.pending()))

## 6. Summary

| | Before | After |
|---|---|---|
| Adding consumer | Edit `BadAgent.run_tool` | `bus.subscribe(new_observer)` |
| Testing Agent | Requires all consumers | Inject empty bus |
| Coupling | Agent → Logger, Metrics, Audit | Agent → EventBus only |
| Open/Closed | ❌ | ✅ |